<a href="https://colab.research.google.com/github/Sagaustus/adh-group-projects/blob/main/group-02-african-languages/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# No Code, No Corpus

### ISO 639-3 as a precondition for computational existence

**Group 2 · working chapter draft**

---

This notebook runs the analysis end to end. Cells marked **YOUR DECISION** are where
your judgement enters, and they are the only part a reader will credit to you.

**The thesis you are testing.** A language's entry into language technology is gated
by an identifier. That identifier is issued almost exclusively to speech varieties
Glottolog has already classified as *languages* rather than *dialects*. So a
classification decision, made on scholarly grounds that are not recorded in this
dataset, determines whether a speech community's language can be computed on at all.

In [ ]:
# Setup — run this first. Nothing to upload.
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

URL = "https://raw.githubusercontent.com/Sagaustus/adh-dh-datasets/main/datasets/08_african_languages/data.csv"
df = pd.read_csv(URL)
print(f"{len(df):,} rows x {df.shape[1]} columns")
print(df.dtypes.to_string())

## Step 1 · Frame

| | Question | Method |
|---|---|---|
| **Descriptive** | How does ISO 639-3 coverage vary across the classification levels Glottolog assigns? | Cross-tabulation with an effect size |
| **Analytical** | What predicts classification as a language rather than a dialect? | Logistic regression — and the answer will surprise you |

The analytical question has an outcome most student projects never reach: a model
that explains almost nothing, for a reason that is itself the finding.

In [ ]:
df["has_iso"] = df["iso639_3"].notna()

print("What one row is:")
print(f"   {len(df):,} entries, {df['family_id'].nunique()} families")
print(f"   levels: {df['level'].value_counts().to_dict()}")
print()
print("ISO 639-3 coverage overall:", f"{df['has_iso'].mean():.1%}")
print()
print("THE GATE — coverage by classification level")
ct = pd.crosstab(df["level"], df["has_iso"])
ct.columns = ["no ISO code", "has ISO code"]
pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)
print(ct.to_string())
print()
print(pct.to_string())

Read that second table before going on.

**94% of things called languages have an ISO code. 1% of things called dialects do.**

The 71.5% of this dataset with no ISO identifier is not a random gap in coverage. It
is, almost exactly, the set of entries Glottolog classified as dialects. One
categorical decision is doing all the work.

## Step 2 · Absence audit

A missing variable, a missing population, and a level of detail too coarse.

In [ ]:
print("WHAT THIS DATASET NEVER RECORDED\n")

print("A missing VARIABLE — the reason")
print("   No column records WHY a variety was classified as a language or a dialect.")
print("   Glottolog cites references for each decision; this export carries none of")
print("   them. The single most consequential field in the dataset is undocumented")
print("   inside it, which is why Step 7 has to leave the table entirely.\n")

print("A missing POPULATION — the unassessed")
print(f"   endangerment recorded for {df['endangerment'].notna().mean():.1%} of entries")
print(f"   documentation recorded for {df['documentation'].notna().mean():.1%}")
print("   An unassessed language is not a safe one. Absence of assessment and")
print("   absence of endangerment are different facts wearing the same blank cell.\n")

print("Detail too COARSE — the country field")
print(f"   `countries` is missing for {df['countries'].isna().mean():.1%} of entries, and")
print("   `n_countries` collapses a language's whole geography to an integer.")
print("   Cross-border speech communities are exactly where the language/dialect")
print("   question is most contested, and that is where the data is thinnest.")

**YOUR DECISION.** Which absence most limits your questions? Two sentences.

Consider that the first absence — no recorded justification — is what forces this
chapter to have a qualitative half at all.

## Step 3 · Describe

The variable at the centre of the question is classification level. Describe it, and
describe what travels with it.

In [ ]:
levels = df["level"].value_counts()
print("classification levels")
for lv, n in levels.items():
    print(f"   {lv:<10}{n:>6,}  ({n/len(df):.1%})")
print()

print("family concentration")
fam = df["family_id"].value_counts()
print(f"   {len(fam)} families; the largest holds {fam.iloc[0]:,} entries "
      f"({fam.iloc[0]/len(df):.0%})")
print(f"   top 3: {fam.head(3).to_dict()}")
print()
print("A count over 'African languages' is substantially a count over one family.")
print("Say so, or a reader will assume the distribution is even.")
print()

print("endangerment, where it is recorded")
print(df["endangerment"].value_counts().to_string())

## Step 4 · Compare

The two groups that matter are the ones the gate separates.

In [ ]:
lang = df[df["level"] == "language"]
dial = df[df["level"] == "dialect"]

print(f"{'':<28}{'language':>12}{'dialect':>12}")
print("-" * 52)
for label, col in [("has ISO 639-3 code", "has_iso"),
                   ("endangerment recorded", "endangerment"),
                   ("documentation recorded", "documentation"),
                   ("country recorded", "countries")]:
    a = lang[col].notna().mean() if col != "has_iso" else lang[col].mean()
    b = dial[col].notna().mean() if col != "has_iso" else dial[col].mean()
    print(f"{label:<28}{a:>11.1%}{b:>12.1%}")
print()
gap = lang["has_iso"].mean() - dial["has_iso"].mean()
print(f"ISO coverage gap: {gap*100:+.1f} percentage points")
print()
print("Write that in percentage points. It is the sentence your chapter turns on.")

## Step 5 · Test

A test **and** an effect size. Then the analytical question — where something
instructive goes wrong.

In [ ]:
from scipy.stats import chi2_contingency

two = df[df["level"].isin(["language", "dialect"])]
table = pd.crosstab(two["level"], two["has_iso"])
chi2, p, dof, expected = chi2_contingency(table)
phi = np.sqrt(chi2 / table.values.sum())

print(f"chi-square {chi2:,.1f}   p = {p:.2e}   phi = {phi:.3f}")
print()
print("phi near 1.0 means the two variables are nearly the same variable.")
print("This is not a subtle association to be teased out — it is a near-identity,")
print("and reporting it as 'statistically significant' would understate it badly.")

### The model that is too good

The obvious next step is to predict classification from everything else in the table.
Run it, and read the diagnostics rather than the score.

In [ ]:
import statsmodels.api as sm

two = df[df["level"].isin(["language", "dialect"])].copy()
two["is_language"] = (two["level"] == "language").astype(int)
fam_size = two["family_id"].value_counts()
two["log_fam"] = np.log(two["family_id"].map(fam_size).fillna(1))
two["documented"] = two["documentation"].notna().astype(int)

X = sm.add_constant(two[["n_countries", "log_fam", "documented"]].astype(float))
naive = sm.Logit(two["is_language"].values, X).fit(disp=0)

print(f"pseudo R-squared = {naive.prsquared:.3f}")
res = pd.DataFrame({"coef": naive.params, "p": naive.pvalues})
res["odds_ratio"] = np.exp(res["coef"])
print(res.to_string(float_format=lambda v: f"{v:.3f}"))
print()
print("An odds ratio in the hundreds and a pseudo R-squared above 0.9 on social data")
print("should stop you, not please you. Something is wrong. The next cell finds it.")

In [ ]:
# Is each covariate DOWNSTREAM of the label we are predicting?
print("Field completeness, by the very label we are trying to predict:\n")
print(f"{'field':<18}{'language':>12}{'dialect':>12}{'family':>12}")
print("-" * 56)
for c in ["countries", "endangerment", "documentation", "iso639_3",
          "latitude", "family_id"]:
    row = df.groupby("level")[c].apply(lambda s: s.notna().mean())
    print(f"{c:<18}{row.get('language',0):>11.0%}{row.get('dialect',0):>12.0%}"
          f"{row.get('family',0):>12.0%}")
print()
print("There it is. Every substantive field is populated almost exclusively for")
print("entries already labelled 'language'. The model was not discovering what")
print("predicts the classification — it was rediscovering the classification through")
print("its own consequences. This is circularity, and it is the chapter's core find:")
print()
print("   the classification decides what gets documented,")
print("   and the documentation is then the evidence for the classification.")

In [ ]:
# The honest model: only fields recorded for EVERY entry, whatever its label.
struct = two.dropna(subset=["latitude", "longitude", "family_id"]).copy()
top_fams = struct["family_id"].value_counts().head(8).index
struct["fam"] = np.where(struct["family_id"].isin(top_fams), struct["family_id"], "other")

Xs = pd.get_dummies(struct[["fam"]], drop_first=True).astype(float)
Xs["latitude"] = struct["latitude"].values
Xs["longitude"] = struct["longitude"].values
Xs = sm.add_constant(Xs)
honest = sm.Logit(struct["is_language"].values, Xs).fit(disp=0)

print(f"STRUCTURAL-ONLY MODEL   n = {len(struct):,}")
print(f"pseudo R-squared = {honest.prsquared:.3f}\n")
r = pd.DataFrame({"coef": honest.params, "p": honest.pvalues})
r["odds_ratio"] = np.exp(r["coef"])
print(r.to_string(float_format=lambda v: f"{v:.3f}"))
print()
print("Geography and family membership explain almost nothing. Latitude and")
print("longitude do not reach significance at all.")
print()
print("That near-zero is a result, not a failed analysis. Whatever decides whether a")
print("variety is a language or a dialect is NOT recorded in this dataset — which is")
print("precisely why Step 7 has to go and read the justifications.")

**YOUR DECISION.** Two things to interpret before moving on.

- One family coefficient is large and highly significant. Find it, and ask what is
  different about that family. (Look at what `sign1238` is.)
- The near-zero pseudo R-squared: is that a limitation of your analysis, or a finding
  about the archive? Argue for one. Your answer determines what §6 of the chapter is
  about.

## Step 6 · Show

One chart. The gate is the finding, so the chart should be the gate — and it should
show the *numbers behind* the proportions, because 1% of 4,206 and 94% of 2,373 are
very different objects.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

order = ["language", "dialect", "family"]
present = [df[df["level"] == lv]["has_iso"].mean() for lv in order]
counts_lv = [int((df["level"] == lv).sum()) for lv in order]

bars = ax1.bar(order, [p * 100 for p in present], color=["#2b5c50", "#9c2c1f", "#855f16"])
for b, pct_, n in zip(bars, present, counts_lv):
    ax1.text(b.get_x() + b.get_width()/2, pct_*100 + 2,
             f"{pct_:.0%}\n(n={n:,})", ha="center", fontsize=9)
ax1.set_ylim(0, 108)
ax1.set_ylabel("entries with an ISO 639-3 code (%)")
ax1.set_title("The gate: coverage by classification level")

stack_have = [df[(df["level"] == lv) & df["has_iso"]].shape[0] for lv in order]
stack_none = [df[(df["level"] == lv) & ~df["has_iso"]].shape[0] for lv in order]
ax2.barh(order, stack_have, color="#2b5c50", label="has ISO code")
ax2.barh(order, stack_none, left=stack_have, color="#d8d3ca", label="no ISO code")
ax2.set_xlabel("entries")
ax2.set_title("The same fact, in counts")
ax2.legend(loc="lower right")

plt.tight_layout()
print(f"CAPTION. ISO 639-3 coverage across all {len(df):,} Glottolog entries for")
print("African languages, by the classification level Glottolog assigns. No filtering")
print("was applied. Left: proportion within each level. Right: absolute counts, which")
print("show that the uncoded majority is composed almost entirely of dialects.")

## Step 7 · The qualitative half

The structural model told you the reason for the classification is not in this table.
So go and find it. This is not an optional enrichment — the quantitative work
established that it is necessary.

The best sample is not random. It is the **boundary cases**: the entries where
Glottolog and ISO 639-3 disagree with each other.

In [ ]:
# The anomalies: dialects that DO have a code, and languages that do NOT.
odd_dialects = df[(df["level"] == "dialect") & df["has_iso"]]
odd_languages = df[(df["level"] == "language") & ~df["has_iso"]]

print(f"dialects WITH an ISO code : {len(odd_dialects):>4}")
print(f"languages WITHOUT one     : {len(odd_languages):>4}")
print()
print("These are the cases where the two systems disagree. Each one is a decision")
print("someone made and defended, and the defence is what you are going to read.\n")

sample = pd.concat([
    odd_dialects.sample(n=min(10, len(odd_dialects)), random_state=3),
    odd_languages.sample(n=min(10, len(odd_languages)), random_state=3),
])[["glottocode", "language_name", "level", "iso639_3", "family_id"]]

print("Twenty to read by hand. Open each at glottolog.org/resource/languoid/id/<code>")
print("and read the references and the classification note:\n")
for _, r in sample.iterrows():
    iso = r["iso639_3"] if pd.notna(r["iso639_3"]) else "—"
    print(f"   https://glottolog.org/resource/languoid/id/{r['glottocode']:<12} "
          f"{str(r['language_name'])[:26]:<28}{r['level']:<10}{iso}")

### The coding scheme

For each of the twenty, record **what basis the classification rests on**. One label
per entry.

| Label | Definition |
|---|---|
| `INTELLIGIBILITY` | Justified by mutual intelligibility evidence, however informal |
| `SOCIOPOLITICAL` | Justified by state, ethnic or national identity |
| `LITERARY` | Justified by a writing tradition, orthography or scripture translation |
| `SOURCE_CONFLICT` | The note records that sources disagree |
| `NO_STATED_BASIS` | A classification with no recorded justification |

**Two coders, working independently.** The category you expect to be rare —
`NO_STATED_BASIS` — is the one to watch. If it is common, the chapter's claim
strengthens considerably.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Replace with your real codes once both coders have worked through the twenty.
coder_a = ["SOCIOPOLITICAL","NO_STATED_BASIS","INTELLIGIBILITY","LITERARY","NO_STATED_BASIS",
           "SOCIOPOLITICAL","SOURCE_CONFLICT","NO_STATED_BASIS","INTELLIGIBILITY","LITERARY",
           "NO_STATED_BASIS","SOCIOPOLITICAL","INTELLIGIBILITY","NO_STATED_BASIS","LITERARY",
           "SOURCE_CONFLICT","NO_STATED_BASIS","SOCIOPOLITICAL","INTELLIGIBILITY","NO_STATED_BASIS"]
coder_b = ["SOCIOPOLITICAL","NO_STATED_BASIS","INTELLIGIBILITY","LITERARY","SOURCE_CONFLICT",
           "SOCIOPOLITICAL","SOURCE_CONFLICT","NO_STATED_BASIS","LITERARY","LITERARY",
           "NO_STATED_BASIS","INTELLIGIBILITY","INTELLIGIBILITY","NO_STATED_BASIS","LITERARY",
           "SOURCE_CONFLICT","NO_STATED_BASIS","SOCIOPOLITICAL","INTELLIGIBILITY","SOCIOPOLITICAL"]

kappa = cohen_kappa_score(coder_a, coder_b)
raw = np.mean([x == y for x, y in zip(coder_a, coder_b)])
print(f"raw agreement {raw:.2f}   Cohen's kappa {kappa:.3f}")
print()
print("Landis & Koch: <0.20 slight, 0.21-0.40 fair, 0.41-0.60 moderate,")
print("0.61-0.80 substantial, >0.80 almost perfect.\n")
from collections import Counter
print("distribution of codes (coder A):", dict(Counter(coder_a)))
print()
print("Disagreements:")
for i, (x, y) in enumerate(zip(coder_a, coder_b), 1):
    if x != y:
        print(f"   entry {i}: {x} vs {y}")
print()
print("A systematic confusion between two labels means the SCHEME needs a sharper")
print("boundary, not that a coder needs correcting. Revise, re-code, report the round.")

## Step 8 · Limits

**What this analysis supports**

- Statements about this Glottolog export and this ISO 639-3 snapshot, on their stated
  dates.
- The near-identity of classification level and ISO coverage, at the effect size
  reported.
- That the fields available here do not predict the classification, and that the
  substantive fields are populated conditionally on it.

**What it does not support**

- That Glottolog's classifications are *wrong*. Nothing here evaluates linguistic
  judgement; the chapter is about what the metadata records and what follows from it.
- Any claim about speakers, communities or language use. This is a catalogue of
  varieties, not of people.
- That ISO 639-3 *causes* exclusion from language technology. You have shown a gate;
  demonstrating the downstream effect needs the model-coverage comparison below.
- Any claim about languages outside Africa, or about Glottolog as a whole.

**YOUR DECISION.** Add two more sentences this data does not support.

In [ ]:
# The downstream test — do this if you want the causal half of the argument.
COVERED_BY_MODELS = {
    "Amharic","Hausa","Igbo","Nigerian Pidgin","Somali","Swahili","Tigrinya","Yoruba",
    "Oromo","Kinyarwanda","Shona","Xhosa","Zulu","Chichewa","Luganda","Setswana","Wolof",
}
have_iso = df[df["has_iso"]]
named = have_iso["language_name"].isin(COVERED_BY_MODELS).sum()

print("From gate to consequence\n")
print(f"   entries in this dataset            {len(df):>8,}")
print(f"   with an ISO 639-3 code             {int(df['has_iso'].sum()):>8,}   "
      f"({df['has_iso'].mean():.0%})")
print(f"   covered by a major African NLP model{named:>7,}   "
      f"({named/len(df):.2%} of all entries)")
print()
print("Each step of that funnel discards an order of magnitude. Report it as a funnel")
print("in your chapter — it is the clearest single piece of evidence you have, and it")
print("takes one table.")
print()
print("To strengthen it: replace COVERED_BY_MODELS above with the published language")
print("lists from the AfriBERTa, AfroXLMR and LaBSE papers, and cite them.")

---

# Step 9 · The chapter template

You have run the analysis. Now you have to write it, and a chapter has a shape.

Fill the gaps below. The sentence frames are there to be used and then rewritten in
your own voice. **Length: 6,000–8,000 words.**

---

## §1 Introduction — *about 800 words*

Problem, claim, why it matters — in that order, and the claim inside the first 200
words.

> Language technology now mediates access to ______________ . Which languages it
> serves is determined less by ______________ than by ______________ .
> This chapter examines ______________ across **_____ Glottolog entries for African
> languages**, and argues that ______________ .

*Write this last.*

---

## §2 The standard and its critics — *about 1,200 words*

What ISO 639-3 is, who maintains it, what Glottolog is, and how they relate. Then the
argument you are joining.

> ISO 639-3 assigns three-letter identifiers to ______________ , maintained by
> ______________ . Glottolog, by contrast, ______________ .
> Scholars have argued ______________ (cite). This chapter contributes
> ______________ .

Name the registration authority and the criteria. "The ISO standard" is not precise
enough for a reviewer.

---

## §3 Data — *about 900 words*

> The dataset comprises **_____ entries** across **_____ families**, classified as
> **_____ languages, _____ dialects and _____ families**. Each row is a languoid.
> Endangerment is recorded for **_____%** and documentation for **_____%**.

Then the absences, as argument:

> Three absences shape the analysis. First, no field records ______________ .
> Second, ______________ . Third, ______________ . The first is decisive, because
> ______________ .

*Feeds in from:* **Steps 1–3**.

---

## §4 Method — *about 700 words*

> Coverage was cross-tabulated against classification level and tested by
> ______________ , reporting ______________ .
> An initial model predicting classification from ______________ was **rejected**,
> because ______________ . A structural model using only ______________ was fitted
> in its place.
> Twenty boundary cases — ______________ and ______________ — were sampled
> purposively and coded independently by two readers.

**Report the rejected model.** A methods section that shows you discarded an
implausible result is more persuasive than one that never mentions it.

*Feeds in from:* **Steps 4–5, 7**.

---

## §5 Findings — *about 1,800 words*

**§5.1 The gate**

> ISO 639-3 coverage is **_____%** among entries classified as languages and
> **_____%** among dialects — a gap of **_____ percentage points**
> (χ² = _____, φ = _____).

**§5.2 The circularity**

> Substantive fields are populated conditionally on the classification: countries for
> **_____%** of languages against **_____%** of dialects, documentation for
> **_____%** against **_____%**. A model using these covariates attains pseudo
> R² = _____ , which reflects ______________ rather than ______________ .

**§5.3 What structure does not explain**

> Using only fields recorded independently of the label — ______________ — the model
> attains pseudo R² = _____ . Geography does not predict classification.
> The exception is ______________ , where ______________ .

**§5.4 What the justifications say** *(qualitative)*

> Twenty boundary cases were coded independently (κ = _____).
> ______ of twenty rested on ______________ , and ______ recorded
> ______________ .

**Figure 1** goes after §5.1, with the caption from Step 6.

---

## §6 Discussion — *about 1,400 words*

> These results suggest ______________ . The classification is therefore best
> understood as ______________ rather than as ______________ .
> For language technology, the consequence is ______________ .

The counter-practice, which is required:

> Initiatives such as ______________ respond by ______________ . Their limits are
> ______________ .

Candidates: Masakhane's community-authored datasets, the ISO 639-3 change-request
process, community-led documentation projects, Local Contexts.

---

## §7 Limitations — *about 500 words*

*Feeds in from:* **Step 8**. Be specific. "This chapter cannot show that ISO coverage
*causes* exclusion; it shows a gate and a funnel" is worth more than a paragraph of
hedging.

---

## §8 Conclusion · §9 References · §10 Data and code availability

> The dataset is available at ______________ . Analysis code is at ______________ .

In [ ]:
# Your numbers, dropped into the frames. Copy out, then rewrite in your own voice.
lv = df["level"].value_counts()
iso_lang = df[df["level"]=="language"]["has_iso"].mean()
iso_dial = df[df["level"]=="dialect"]["has_iso"].mean()

print("=" * 74)
print("DRAFT SENTENCES — your numbers already placed")
print("=" * 74)

print("""
§3 DATA
  The dataset comprises {n:,} Glottolog entries for African languages across {f}
  families, classified as {lang:,} languages, {dia:,} dialects and {fam:,} families.
  Each row is a languoid. An ISO 639-3 identifier is present for {iso:.1%} of
  entries, endangerment status for {end:.1%} and documentation status for {doc:.1%}.
  No field records the basis on which a variety was classified.
""".format(n=len(df), f=df["family_id"].nunique(), lang=int(lv.get("language",0)),
           dia=int(lv.get("dialect",0)), fam=int(lv.get("family",0)),
           iso=df["has_iso"].mean(), end=df["endangerment"].notna().mean(),
           doc=df["documentation"].notna().mean()))

print("""§5.1 THE GATE
  ISO 639-3 coverage is {a:.1%} among entries classified as languages and {b:.1%}
  among those classified as dialects, a gap of {g:.1f} percentage points
  (chi-square = {c:,.0f}, phi = {p:.3f}). The identifier tracks the classification
  almost exactly.
""".format(a=iso_lang, b=iso_dial, g=(iso_lang-iso_dial)*100, c=chi2, p=phi))

print("""§5.2 THE CIRCULARITY
  Substantive fields are populated conditionally on the classification being
  predicted: country is recorded for {c1:.0%} of languages against {c2:.0%} of
  dialects, and documentation for {d1:.0%} against {d2:.0%}. A model using these
  covariates attains pseudo R-squared {r:.3f}, which reflects the archive's own
  recording practice rather than any independent predictor of classification.
""".format(c1=df[df.level=="language"]["countries"].notna().mean(),
           c2=df[df.level=="dialect"]["countries"].notna().mean(),
           d1=df[df.level=="language"]["documentation"].notna().mean(),
           d2=df[df.level=="dialect"]["documentation"].notna().mean(),
           r=naive.prsquared))

print("""§5.3 WHAT STRUCTURE DOES NOT EXPLAIN
  Using only coordinates and family membership - the fields recorded independently
  of the label - the model attains pseudo R-squared {r:.3f} (n = {n:,}). Latitude and
  longitude do not reach conventional significance. Whatever grounds the
  language/dialect distinction is not recorded in this dataset.
""".format(r=honest.prsquared, n=len(struct)))

print("""§5.4 QUALITATIVE
  Twenty boundary cases - {od} dialects holding an ISO code and {ol} languages
  lacking one - were coded independently by two readers (kappa = {k:.3f}).
""".format(od=len(odd_dialects), ol=len(odd_languages), k=kappa))
print("=" * 74)
print("Left for you: what it MEANS, what it cannot support, and the counter-practice.")

### Three mistakes that sink first chapters

**Reporting the rejected model as a finding.** Pseudo R² of 0.9 on social data is a
symptom. A chapter that reports it proudly will be caught; one that reports it as a
diagnostic looks rigorous.

**Treating the near-zero structural model as failure.** It is the result that makes
the qualitative section necessary. Frame it that way in §5.3 and §6.

**Claiming causation from the funnel.** You have a gate and a coincidence of coverage.
The causal claim needs the model-coverage comparison and a citation, and even then it
is an argument, not a demonstration.

---

# Step 10 · Dividing the work

More than five people, one chapter. Divide by **expertise**, not by paragraph count —
a chapter where everyone wrote a bit of everything reads like it.

## The roles

| # | Role | Owns | Expertise it draws on | Hands over |
|---|---|---|---|---|
| 1 | **Corpus &amp; provenance** | §3 Data, absence audit | Library and archival science; knowledge of Glottolog | A described dataset and a defended list of absences |
| 2 | **Analysis** | §4 Method, §5.1–5.3 | Statistics, computation | The gate, the rejected model, the structural model |
| 3 | **Coder A** | §5.4, jointly | Linguistics; close reading of classification notes | 20 independently assigned codes |
| 4 | **Coder B** | §5.4, jointly | Linguistics; close reading | 20 independently assigned codes |
| 5 | **Theory &amp; literature** | §2, §6 | Sociolinguistics, postcolonial studies, language policy | The argument the chapter joins, and the counter-practice |
| 6 | **NLP &amp; consequence** | the funnel, §6 | Language technology | Published model language lists, cited |
| 7 | **Integration editor** | §1, §7, references | Editorial judgement | One voice, and a chapter that ends |

Role 6 is specific to this group: somebody has to go to the AfriBERTa, AfroXLMR and
LaBSE papers and extract their actual language lists. That is the difference between
asserting the funnel and demonstrating it.

## Why coders 3 and 4 are two people

A **method requirement, not a staffing convenience.** Cohen's kappa measures whether
two readers applying the same scheme independently reach the same judgement. One
person coding all twenty leaves §5.4 with no evidential standing.

Agree the scheme, separate, code, then compare. Do not discuss the entries while
coding.

## The order things happen in

```
   Corpus & provenance ──┐
                         ├──► Analysis ──┐
   Coders A + B ─────────┘               ├──► Integration
                                         │
   Theory & literature ──────────────────┤
   NLP & consequence  ───────────────────┘
```

Theory and the NLP funnel can both start on day one. Integration cannot start until
everything else exists, and the introduction is written last.

## Combining the drafts

**One person edits for voice, and everyone accepts the edit.** Not by vote.

**Agree the terms in writing before drafting.** For this chapter that means: do you
say *languoid*, *variety* or *entry*? Is it *the classification* or *the level*? Ten
minutes now saves a day later.

## Declaring who did what

Use **CRediT** — conceptualisation, data curation, formal analysis, investigation,
methodology, software, visualisation, writing – original draft, writing – review and
editing. In interdisciplinary work it is how the linguist and the statistician each
get credited for what they actually did.

In [ ]:
TEAM = {
    "Corpus & provenance":   ("________________", "data curation, investigation"),
    "Analysis":              ("________________", "formal analysis, software, methodology"),
    "Coder A":               ("________________", "investigation, validation"),
    "Coder B":               ("________________", "investigation, validation"),
    "Theory & literature":   ("________________", "conceptualisation, writing - original draft"),
    "NLP & consequence":     ("________________", "investigation, data curation"),
    "Integration editor":    ("________________", "writing - review and editing, supervision"),
}
print("AUTHOR CONTRIBUTIONS (CRediT)")
print()
for role, (name, credit) in TEAM.items():
    print(f"  {name}: {credit}.")
    print(f"      [{role}]")
print()
print("Fill in names, delete the bracketed labels, place after the conclusion.")
print("Agree authorship order EARLY - it is the argument that ruins collaborations")
print("when left to the end.")

### What goes wrong, and how to see it coming

**Everyone waits for the analysis.** Roles 1, 5 and 6 can all start immediately.

**The coders talk.** Fatal to the kappa. If it happens, say so in §4 rather than
reporting a figure you know is inflated.

**The funnel gets asserted rather than sourced.** Role 6 must cite the actual model
papers. "Major models cover few African languages" without a citation is the kind of
sentence a reviewer will ask you to remove.

**Nobody owns the ending.** Role 7 owns §8 and owns the decision that the draft is
finished.

## What to hand in

1. **One page**: the finding, the method, the uncertainty, and the limits list.
2. **One chart**, captioned, saying what you filtered.
3. **The coding sheet** with both coders' labels and the kappa.

### Turning this into the chapter

The spine is unusually strong here: a near-deterministic gate, a demonstrated
circularity, a structural model that explains nothing, and a funnel with a
consequence. What it still needs from you is **§6** — what it means that a
classification made on unrecorded grounds decides which languages can be computed on
at all — and the counter-practice. Masakhane is the obvious place to look, and its
data was built by exactly the communities this gate excludes.